all dependables

In [53]:
import pymol
from pymol import cmd
import csv
import os
from collections import defaultdict
import pandas as pd # Using pandas makes reading CSVs much cleaner
from Bio.PDB import PDBParser
from tqdm import tqdm

# 1. Getting peptide contact using distance

In [45]:
def pdb_interface_distance(pdb_file, peptide_chain, protein_chain, cutoff=4.0, export_csv = True):
    '''Worker function to collect all interacting Mpro residues to peptide'''
    # Load the PDB file into PyMOL
    # 'object_name' will be the filename without the .pdb extension
    subsite_map = defaultdict(set)
    object_name = os.path.basename(pdb_file).split('.')[0]
    
    try:
        cmd.load(pdb_file, object_name)
    
        # Define our selection strings based on the chains provided
        peptide_sel = f"{object_name} and chain {peptide_chain}"
        protein_sel = f"{object_name} and chain {protein_chain}"
        output_csv = f"{object_name}_interactions.csv"

        # Get peptide residues
        peptide_residues = []
        cmd.iterate(f"{peptide_sel} and name CA", 
                "peptide_residues.append((int(resi), resn))", 
                space={'peptide_residues': peptide_residues})
    
        # find the GLN (P1)
        p1_resi = next((resi for resi, resn in peptide_residues if resn == "GLN"), None)

        if p1_resi is None:
            print(f"! No GLN found in peptide {object_name}")
            return {}

        # storing it in Schecher-Berger Notation
        label_map = {}
        for resi, resn in peptide_residues:
            diff = resi - p1_resi
            if diff == 0: label = "P1"
            elif diff == -1: label = "P2"
            elif diff == -2: label = "P3"
            elif diff == -3: label = "P4"
            elif diff == -4: label = "P5"
            elif diff == -5: label = "P6"
            elif diff == 1: label = "P1_prime"
            elif diff == 2: label = "P2_prime"
            elif diff == 3: label = "P3_prime"
            elif diff == 4: label = "P4_prime"
            elif diff == 5: label = "P5_prime"
            elif diff == 6: label = "P6_prime"
            else: continue
            label_map[resi] = label

        #iterate through structure to find nearby interactions
        for resi, label in label_map.items():
            current_res_sel = f"({peptide_sel} and resi {resi})"
            interaction_sel = f"({protein_sel}) and (all within {cutoff} of {current_res_sel})"

            interacting_res_list = []
            # We use 'byres' to get the full identity of the neighbor
            cmd.iterate(f"byres {interaction_sel}", 
                        "interacting_res_list.append(resi)", 
                        space={'interacting_res_list': interacting_res_list})

            subsite_label = label.replace("P", "S")
            subsite_map[subsite_label].update({int(r) for r in interacting_res_list})

        #write csv if export_csv = true
        if export_csv:
            output_csv = f"{object_name}_subsite_map.csv"

            with open(output_csv, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["Subsite", "Mpro_Resi"])

                for subsite, m_res in subsite_map.items():
                    interactions_str = ";".join(str(r) for r in sorted(m_res))
                    writer.writerow([subsite, interactions_str])
            print(f"Success! Created {output_csv}")


    except Exception as e:
        print(f"! ERROR: {e}")

    finally:
        cmd.delete(object_name)

    return subsite_map


In [119]:
pdb_interface_distance("Mpro-nsp/7TA4.pdb", "D", "B")
#in the paper they used 3.6 Angstroms as the distance they used to detect H bonds of protons to any heavy atom

Success! Created 7TA4_subsite_map.csv


defaultdict(set,
            {'S6': {168, 191, 645},
             'S5': {168, 189, 190, 453},
             'S4': {165, 166, 167, 188, 189, 190, 192},
             'S3': {165, 166, 189, 722},
             'S2': {41, 49, 164, 165, 187, 188, 189, 517},
             'S1': {41,
              140,
              141,
              142,
              143,
              144,
              145,
              163,
              164,
              165,
              166,
              172,
              531},
             'S1_prime': {25, 26, 41, 142, 143, 145},
             'S2_prime': {24, 25, 26, 143, 616},
             'S3_prime': {24, 26, 444, 464, 640},
             'S4_prime': {24, 26, 464, 689}})

# 2. Generate consensus subsite based on distances

Now generate for all 9 nsp found in the literature - might expand on the literature later to include all nsps for mpro
- only append to csv if the residue appears in more than 3 nsps
- then manually add on other missing residues that have been reported in the literature to make important interactions (make a record of that)

In [47]:
# --- CONFIGURATION / CONSTANTS ---
# This is the "Source of Truth" for your UMAP and CSV order
BIOLOGICAL_ORDER = [
    "S6", "S5", "S4", "S3", "S2", "S1", 
    "S1_prime", "S2_prime", "S3_prime", "S4_prime", "S5_prime", "S6_prime"
]

# --- UTILITIES ---
def subsite_sort_key(label):
    """Utility to map subsite strings to numerical ranks for sorting."""
    try:
        return BIOLOGICAL_ORDER.index(label)
    except ValueError:
        return 999  # Catch-all for unexpected labels

In [48]:

def subsite_consensus_collector(csv_path, pdb_dir, dist_cutoff=4.0, min_nsps=3, export_csv=False):
    """
    COLLECTOR: Reads the chain_id CSV and aggregates subsite data.
    """
    # Master structure: master_bin[subsite][mpro_resi] = [list of nsp names]
    master_bin = defaultdict(lambda: defaultdict(list))
    
    # 1. Load your chain metadata
    df = pd.read_csv(csv_path)
    
    print(f"Starting analysis on {len(df)} NSP complexes...")

    for _, row in df.iterrows():
        nsp_label = row['nsp']
        pdb_code = row['pdb_code']
        pep_chain = row['peptide_chain']
        prot_chain = row['protein_chain']
        
        # Construct path (assumes files are named e.g., '7T70.pdb')
        pdb_path = os.path.join(pdb_dir, f"{pdb_code}.pdb")
        
        if not os.path.exists(pdb_path):
            print(f" ! File missing: {pdb_path}")
            continue
            
        print(f" - Processing {nsp_label} ({pdb_code})...")
        
        # Call the Anchor-based Worker
        # Set export_csv=True if you want the individual mapping files for each NSP
        nsp_results = pdb_interface_distance(pdb_path, pep_chain, prot_chain, 
                                                 cutoff=dist_cutoff, export_csv=False)
        
        # 2. Bin the results into the master dictionary
        for subsite, mpro_residues in nsp_results.items():
            for resi in mpro_residues:
                master_bin[subsite][resi].append(nsp_label)

    # 3. Generation of conflict checker
    resi_to_subsite_count = defaultdict(set)
   
    for subsite, resi_dict in master_bin.items():
        for resi, nsp_list in resi_dict.items():
            resi_to_subsite_count[resi].add(subsite)

    # 4. generate consensus sequence and export_csv if that is set to true
    consensus_definitions = defaultdict(list)
    sorted_subsites = sorted(master_bin.keys(), key=subsite_sort_key)
    
    for subsite in sorted_subsites:
        for resi in sorted(master_bin[subsite].keys()):
            nsps = master_bin[subsite][resi]
            if len(nsps) >= min_nsps:
                # This runs regardless of the CSV flag
                consensus_definitions[subsite].append(resi)

    #manually append Ser1 to the subsite list at S1 position - biological significance
    if 1 not in consensus_definitions["S1"]:
        consensus_definitions["S1"].append(1)
        print("   + Manually appended Ser1 to S1 subsite")

    consensus_definitions = dict(sorted(consensus_definitions.items(), key=lambda x: subsite_sort_key(x[0])))

    if export_csv:
        output_csv = f"mpro_consensus_subsites_{dist_cutoff}A_{min_nsps}.csv"
        with open(output_csv, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Subsite", "Mpro_Resi", "NSP_Count", "NSPs", "Conflict_Warning"])

            for subsite, residues in consensus_definitions.items():
                for resi in residues:
                    nsps = master_bin[subsite][resi]
                    other_subsites = resi_to_subsite_count[resi] - {subsite}
                    warning = f"Also in {list(other_subsites)}" if other_subsites else ""
                    writer.writerow([subsite, resi, len(nsps), "; ".join(nsps), warning])
    
        print("-" * 30)
        print(f"Success! Consensus results saved to {output_csv}")

    return dict(consensus_definitions)

# Example Run:
# subsite_consensus_collector("chain_id.csv", "./pdb_folder", dist_cutoff=4.0)

In [49]:
pdb_dir = "Mpro-nsp"
chain_dir = "Mpro-nsp/chain_id.csv"

consensus_subsuite = subsite_consensus_collector(chain_dir, pdb_dir, dist_cutoff=3.8)

Starting analysis on 9 NSP complexes...
 - Processing nsp 4-5 (7T70)...
 - Processing nsp 5-6 (7T8M)...
 - Processing nsp 7-8 (7T8R)...
 - Processing nsp 8-9 (7T9Y)...
 - Processing nsp 9-10 (7TA4)...
 - Processing nsp 10-11 (7TA7)...
 - Processing nsp 12-13 (7TB2)...
 - Processing nsp 13-14 (7TBT)...
 - Processing nsp 15-16 (7TC4)...
   + Manually appended Ser1 to S1 subsite


Going with 3.8 Angstorm as the distance measurement don't need to append anything in the interaction pattern
- what about Ser1 in the other chain? - manually added

# 3. Generate csv for fragalysis SARS-CoV-2 Mpro metadata
- required for ATOMICA
- and binning based on distance

call it parse_fragalysis_metadata 

In [40]:
def parse_fragalysis_metadata(metadata_path, pdb_dir, dimer_info = False):
    '''CSV file containing the following headers [ pdb_id | pdb_path | chain1 | chain2 | lig_code | lig_smiles | lig_resi ] 
    Make sure pdb_dir is from the root so it can be accessed anywhere
    if dimer_info = True - output - fragalysis_parsed_full_(length of csv).csv => this is for downstream analysis
    or dimer_info = False - output - atomica_index_file_(length of csv).csv => this is intended to be fed into ATOMICA'''

    df = pd.read_csv(metadata_path)
    print("Starting to parse the fragalysis metadata")

    rows = []

    for _, row in df.iterrows():
        long_code = row['Long code']

        #skip the MERS data
        if long_code.startswith("MERS"):
            continue

        smiles = row['Smiles']
        pose = row['Pose']

        #skip missing data and report it
        if pd.isna(pose) or pd.isna(long_code):
            print(f"  ! Skipping row with {pose} (pose) or {long_code} (long_code) with missing data")
            continue

        #extract chain id and lig_resi number from long code
        parts = long_code.split("_")
        chain = parts[-4]
        lig_resi = parts[-3]

        #extract pdb name to file path 
        if chain == "A":
            pdb_id = f"{pose}.pdb"
            pdb_path = os.path.join(pdb_dir, pdb_id)
        else:
            pdb_id = f"{pose[:-1]}b.pdb"
            pdb_path = os.path.join(pdb_dir, pdb_id)
        
        #check whether the pdb path actually exists
        if not os.path.exists(pdb_path):
            pdb_id = f"{pose[:-1]}a.pdb"
            fallback_path = os.path.join(pdb_dir, pdb_id)

            if os.path.exists(fallback_path):
                pdb_path = fallback_path
            else:
                print(f"  ! File missing: {pdb_path}")
                continue
        
        row_data = {
                "pdb_id": pdb_id,
                "pdb_path": pdb_path,
                "chain1": chain,
                "chain2": chain,
                "lig_code": "LIG",
                "lig_smiles": smiles,
                "lig_resi": lig_resi
            }
        
        #allow keeping of the quaternay_information
        if dimer_info:
            row_data["quaternary_assembly"] = row['Quatassemblies upload name']

        rows.append(row_data)
        

    output_df = pd.DataFrame(rows)
    
    if dimer_info:
        output_csv = f"fragalysis_parsed_full_{len(output_df)}.csv"
    else:
        output_csv = f"atomica_index_file_{len(output_df)}.csv"
    

    if not output_df.empty:
        output_df.to_csv(output_csv, index=False)
        print("-" * 30)
        print(f"Success! {len(output_df)} complexes saved to {output_csv}")
    else:
        print("! No valid complexes found - CSV not created")


In [ ]:
metadata_path = "../metadata.csv"
pdb_dir = "/Users/hannah/code/rotation1/CoV-Mpro/SARS_Mpro"

parse_fragalysis_metadata(metadata_path, pdb_dir, dimer_info=True) ##DO NOT RUN ANYTHING AGAIN 
#AS IT WILL OVERWRITE THE MANUAL CHANGE at 7gfz-b.pdb

Starting to parse the fragalysis metadata
  ! Skipping row with nan (pose) or 7gmf_A_408_x_v1 (long_code) with missing data
------------------------------
Success! 1319 complexes saved to fragalysis_parsed_full_1319.csv


In [ ]:
#check for duplicates
df = pd.read_csv("fragalysis_index_file_1319.csv")
duplicates = df[df['pdb_id'].duplicated(keep=False)]
print(duplicates[['pdb_id', 'chain2', 'lig_resi']])

#manually changed 778 to 7gfz-b.pdb (LIG position 405)

         pdb_id chain2  lig_resi
777  7gfz-a.pdb      A       404
778  7gfz-a.pdb      A       405


# 4. Bining the Fragalysis substrates

In [31]:
#trying biopython

from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure("7gef-a", "/Users/hannah/code/rotation1/CoV-Mpro/SARS_Mpro/7gef-a.pdb")

for atom in structure.get_atoms():
    residue = atom.get_parent()
    chain = residue.get_parent()

    print(chain.id, residue.id[1], residue.resname, atom.name)

A 1 SER N
A 1 SER CA
A 1 SER C
A 1 SER O
A 1 SER CB
A 1 SER OG
A 2 GLY N
A 2 GLY CA
A 2 GLY C
A 2 GLY O
A 3 PHE N
A 3 PHE CA
A 3 PHE C
A 3 PHE O
A 3 PHE CB
A 3 PHE CG
A 3 PHE CD1
A 3 PHE CD2
A 3 PHE CE1
A 3 PHE CE2
A 3 PHE CZ
A 4 ARG N
A 4 ARG CA
A 4 ARG C
A 4 ARG O
A 4 ARG CB
A 4 ARG CG
A 4 ARG CD
A 4 ARG NE
A 4 ARG CZ
A 4 ARG NH1
A 4 ARG NH2
A 5 LYS N
A 5 LYS CA
A 5 LYS C
A 5 LYS O
A 5 LYS CB
A 5 LYS CG
A 5 LYS CD
A 5 LYS CE
A 5 LYS NZ
A 6 MET N
A 6 MET CA
A 6 MET C
A 6 MET O
A 6 MET CB
A 6 MET CG
A 6 MET SD
A 6 MET CE
A 7 ALA N
A 7 ALA CA
A 7 ALA C
A 7 ALA O
A 7 ALA CB
A 8 PHE N
A 8 PHE CA
A 8 PHE C
A 8 PHE O
A 8 PHE CB
A 8 PHE CG
A 8 PHE CD1
A 8 PHE CD2
A 8 PHE CE1
A 8 PHE CE2
A 8 PHE CZ
A 9 PRO N
A 9 PRO CA
A 9 PRO C
A 9 PRO O
A 9 PRO CB
A 9 PRO CG
A 9 PRO CD
A 10 SER N
A 10 SER CA
A 10 SER C
A 10 SER O
A 10 SER CB
A 10 SER OG
A 11 GLY N
A 11 GLY CA
A 11 GLY C
A 11 GLY O
A 12 LYS N
A 12 LYS CA
A 12 LYS C
A 12 LYS O
A 12 LYS CB
A 12 LYS CG
A 12 LYS CD
A 12 LYS CE
A 12 LYS NZ
A 13 V

## 4.1 get Mpro residues that are in contact with the ligand within the fragalysis datasets (filtered to SARS-CoV-2 Mpro)

In [32]:
def get_contact_distance(pdb_id, pdb_path,lig_chain, lig_resi, cutoff = 3.8):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)

    contacted_Mpro_residue = set()
    lig_atoms = []
    protein = defaultdict(list)

    for atom in structure.get_atoms():
        residue = atom.get_parent()
        chain = residue.get_parent()

        if chain.id == lig_chain and residue.id[1] == int(lig_resi):
            lig_atoms.append(atom)
        else:
            protein[residue.id[1]].append(atom)
    
    for l_atom in lig_atoms:
        for resi, prot_atoms in protein.items():
            for prot_atom in prot_atoms:
                distance = l_atom - prot_atom
                if distance <= cutoff:
                    contacted_Mpro_residue.add(resi)

    return list(contacted_Mpro_residue)

In [33]:
pdb_id = '7gef-a.pdb'
pdb_path = "/Users/hannah/code/rotation1/CoV-Mpro/SARS_Mpro/7gef-a.pdb"

get_contact_distance(pdb_id, pdb_path, lig_chain="A", lig_resi="403")

[163,
 739,
 164,
 166,
 165,
 867,
 41,
 586,
 140,
 187,
 141,
 142,
 145,
 601,
 507,
 188,
 189]

In [54]:
def get_ligand_contacts(pdb_index_csv, cutoff = 3.8):
    '''get all fragalysis Mpro complex's ligand interaction Mpro residues
    pdb_index_csv is the output from parse_fragalysis_metadata'''
    df = pd.read_csv(pdb_index_csv)

    ligand_contacts = defaultdict(list)

    print("Start Collecting the residues that interacts with the ligands in fragalysis")

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Getting ligand contacts"):
        pdb_id = row["pdb_id"]
        pdb_path = row["pdb_path"]
        lig_chain = row["chain2"]
        lig_resi = row["lig_resi"]
    
        contacted_residues = get_contact_distance(pdb_id=pdb_id, pdb_path=pdb_path, lig_chain=lig_chain, lig_resi=lig_resi, cutoff=cutoff)
        ligand_contacts[pdb_id].extend(contacted_residues)
    
    print(f"Success! {len(ligand_contacts)} of complexes analysed and stored.")
    
    return ligand_contacts

In [ ]:
#test to see if it works
df = pd.read_csv("fragalysis_index_file_1319.csv")
df.head(5).to_csv("test_5.csv", index=False)

result = get_ligand_contacts("test_5.csv", cutoff=3.8)
print(result)
len(result)

defaultdict(<class 'list'>, {'SARS-b0110a.pdb': [163, 165, 166, 485, 41, 362, 140, 141, 142, 144, 49, 466, 145, 469, 471, 412, 316, 157], 'SARS-b0110b.pdb': [481, 163, 355, 165, 166, 41, 140, 142, 464, 49, 145, 210, 189], 'SARS-b0111a.pdb': [163, 165, 166, 41, 140, 141, 142, 144, 49, 188, 145, 306, 189, 153, 187, 604, 605], 'SARS-b0111b.pdb': [140, 141, 142, 144, 145, 158, 163, 164, 165, 166, 41, 429, 49, 187, 188, 189, 202, 333, 600, 502], 'SARS-b0112a.pdb': [163, 165, 166, 41, 140, 525, 302, 526, 142, 49, 145, 403, 157, 537, 187, 188, 189]})


5

In [50]:
def bin_complexes_to_subsites(fragalysis_pdb_index_csv, consensus_subsuite_definition, cutoff = 3.8, output_path = None):
    #run the two functions predefined
    consensus_subsite = consensus_subsuite_definition
    ligand_contacts = get_ligand_contacts(fragalysis_pdb_index_csv, cutoff=cutoff)

    ligand_subsite_assignment = defaultdict(set)

    for fragalysis_pdb, residues in ligand_contacts.items():
        for resi in residues:
            for mpro_subsites, cons_resi in consensus_subsite.items():
                if resi in cons_resi:
                    ligand_subsite_assignment[fragalysis_pdb].add(mpro_subsites)
    
    if output_path:
        df = pd.read_csv(output_path)
        df["subsites"] = df["pdb_id"].map(lambda x: list(ligand_subsite_assignment.get(x, set())))
        df.to_csv(output_path, index=False)
        print(f"substie information successfully added to {output_path} under the title")

    return ligand_subsite_assignment


In [55]:
fragalysis_prased_csv = "fragalysis_parsed_full_1319.csv"

#consensus_subsite_definition is defined in 2. named consensus_subsite

bin = bin_complexes_to_subsites(fragalysis_prased_csv, consensus_subsuite)

Start Collecting the residues that interacts with the ligands in fragalysis


Getting ligand contacts: 100%|██████████| 1319/1319 [03:40<00:00,  5.97it/s]

Success! 1319 of complexes analysed and stored.


In [61]:
df = pd.read_csv("fragalysis_parsed_full_1319.csv")
df["subsites"] = df["pdb_id"].map(lambda x: list(bin.get(x, set())))
df["subsites"] = df["subsites"].apply(lambda x: x if x else ["others"])
df.to_csv("fragalysis_parsed_full_1319.csv", index=False)

In [ ]:
df["subsites"].apply(lambda x: x == ["others"]).sum() 

#29 dropouts (no interaction with key subsite residues on mpro) now labelled as others in subsite section

29

In [64]:
print(df["subsites"].head(10).tolist())
print(df["subsites"].dtype)

[['S1_prime', 'S1', 'S2', 'S3', 'S4'], ['S1_prime', 'S1', 'S2', 'S3', 'S4', 'S5'], ['S1_prime', 'S1', 'S2', 'S3', 'S4', 'S5'], ['S1_prime', 'S1', 'S2', 'S3', 'S4', 'S5'], ['S1_prime', 'S1', 'S2', 'S3', 'S4', 'S5'], ['S1_prime', 'S1', 'S2', 'S3', 'S4', 'S5'], ['S1_prime', 'S1', 'S2', 'S3', 'S4'], ['S1_prime', 'S1', 'S2', 'S3', 'S4'], ['S1_prime', 'S1', 'S2', 'S3', 'S4', 'S5'], ['S1_prime', 'S1', 'S2', 'S3', 'S4', 'S5']]
object
